# Дополнительный разбор AI4I: классификация

**Производственная задача:** ранжировать детали по риску отказа.  
**Результат:** честно сравнить четыре baseline-модели, выбрать рабочую и собрать submission.

Перед запуском сформулируйте вслух: объект, момент решения, target, горизонт прогноза и действие после score.

Необязательный кейс для самостоятельного изучения. Основная практика — 35 минут (25 минут baseline + 10 минут эксперимент): `../competition/baseline_regression.ipynb`. Файлы этого кейса не предназначены для All Cups 3521.


## 0. Контракт задачи

- Объект: одна деталь оборудования.
- Target: `machine_failure`.
- Основная метрика сравнения: ROC-AUC; дополнительная — PR-AUC из-за редкого положительного класса.
- Test используется только для итогового предсказания.

**Вопрос:** какой столбец был бы утечкой, если его можно узнать только после отказа?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

def find_package_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "train.csv").exists():
            return candidate
    raise FileNotFoundError("Запустите ноутбук из supplementary_ai4i_case_study/ или его подкаталога")

PACKAGE_DIR = find_package_dir()
TRAIN_PATH = PACKAGE_DIR / "data" / "train.csv"
TEST_PATH = PACKAGE_DIR / "data" / "test.csv"
SAMPLE_PATH = PACKAGE_DIR / "data" / "sample_submission.csv"
PACKAGE_DIR

PosixPath('/Users/artyem.petrov/dev/teaching/miem-hse-h2-2026/teachers/lesson_04_supervised_learning/supplementary_ai4i_case_study')

## 1. Данные и sanity checks

Сначала проверяем не модель, а контракт данных: ключ, target, типы и долю отказов.

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

TARGET = "machine_failure"
ID_COL = "part_id"

assert TARGET in train and TARGET not in test
assert train[ID_COL].is_unique and test[ID_COL].is_unique
assert set(train[TARGET].unique()) <= {0, 1}

print("train:", train.shape, "test:", test.shape)
print("Доля отказов:", f"{train[TARGET].mean():.2%}")
display(train.head())
display(train.dtypes.to_frame("dtype"))

train: (7500, 8) test: (2500, 7)
Доля отказов: 3.39%


,part_id,part_type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure
0,PART-00001,L,301.516,310.689,1689.0,27.877,135.0,0
1,PART-00002,M,303.032,312.485,1601.0,33.777,135.0,0
2,PART-00003,L,NaN,313.700,1509.0,35.021,205.0,1
3,PART-00004,M,302.583,310.199,1561.0,38.983,97.0,0
4,PART-00005,L,297.576,309.490,1551.0,34.896,68.0,0


,dtype
part_id,str
part_type,str
air_temperature_k,float64
process_temperature_k,float64
rotational_speed_rpm,float64
torque_nm,float64
tool_wear_min,float64
machine_failure,int64


## 2. Один preprocessing — разные семейства моделей

Предобработка находится **внутри Pipeline**, поэтому imputer, scaling и one-hot обучаются только на train-части каждого фолда.

In [3]:
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

categorical_features = X.select_dtypes(exclude="number").columns.tolist()
numeric_features = X.columns.difference(categorical_features).tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

models = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Tree": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Forest": RandomForestClassifier(
        n_estimators=160,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

def make_model(estimator):
    return Pipeline([("preprocess", preprocess), ("model", estimator)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

## 3. Честное сравнение baseline-моделей

Все модели получают одни и те же фолды и метрики. Смотрите не только на среднее, но и на разброс.

Выбираем модель по среднему ROC-AUC. PR-AUC используем для дополнительной диагностики и разрешения равенства ROC-AUC.


In [4]:
rows = []
for name, estimator in models.items():
    scores = cross_validate(
        make_model(estimator), X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False
    )
    rows.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "pr_auc_mean": scores["test_pr_auc"].mean(),
        "pr_auc_std": scores["test_pr_auc"].std(),
    })

cv_results = pd.DataFrame(rows).sort_values(["roc_auc_mean", "pr_auc_mean"], ascending=False).reset_index(drop=True)
display(cv_results.style.format({c: "{:.3f}" for c in cv_results.columns if c != "model"}))

best_name = cv_results.loc[0, "model"]
print("Рабочая модель по ROC-AUC:", best_name)

,model,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Forest,0.969,0.015,0.698,0.044
1,Tree,0.929,0.024,0.411,0.058
2,Logistic,0.898,0.012,0.428,0.048
3,Dummy,0.500,0.000,0.034,0.000


Рабочая модель по ROC-AUC: Forest


### Контрольная точка

1. Какая модель сильнее Dummy?
2. Что изменилось в предположениях о данных от Logistic к Tree и Forest?
3. Есть ли модель с высоким средним, но нестабильными фолдами?

## 4. Метрика сравнивает, порог задаёт действие

Получим out-of-fold score выбранной модели. Ни одна строка не оценивается моделью, которая видела её при обучении.

$$\mathrm{Precision}=\frac{TP}{TP+FP},\qquad
\mathrm{Recall}=\frac{TP}{TP+FN}.$$

Положительный класс — отказ. Решение $\widehat y=\mathbf 1\{p\geq\tau\}$ зависит
от порога $\tau$, даже когда вероятности $p$ остались прежними. При нулевом
знаменателе код явно использует `zero_division=0`.


In [5]:
chosen = make_model(models[best_name])
oof_score = cross_val_predict(
    chosen, X, y, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

threshold_rows = []
for threshold in [0.20, 0.35, 0.50, 0.65]:
    pred = (oof_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "false_alarms": fp,
        "missed_failures": fn,
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.style.format({"precision": "{:.3f}", "recall": "{:.3f}"}))

,threshold,precision,recall,false_alarms,missed_failures
0,0.200000,0.362,0.854,382,37
1,0.350000,0.548,0.764,160,60
2,0.500000,0.705,0.602,64,101
3,0.650000,0.835,0.358,18,163


### Решение в парах

Выберите порог для ситуации «пропущенный отказ намного дороже проверки». Объясните выбор через число пропусков и ложных тревог, а не через слово «лучше».

## 5. Итоговая модель и submission

После выбора семейства и проверки процесса обучаемся на всём train. Test не используется ни для выбора модели, ни для порога.

In [6]:
final_model = make_model(models[best_name])
final_model.fit(X, y)
test_score = final_model.predict_proba(X_test)[:, 1]

submission = pd.read_csv(SAMPLE_PATH)
assert submission[ID_COL].tolist() == test[ID_COL].tolist()
submission[TARGET] = test_score

OUTPUT_PATH = PACKAGE_DIR / "seminar_submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)

assert len(submission) == len(test)
assert submission[TARGET].between(0, 1).all()
print("Сохранено:", OUTPUT_PATH)
display(submission.head())

Сохранено: /Users/artyem.petrov/dev/teaching/miem-hse-h2-2026/teachers/lesson_04_supervised_learning/supplementary_ai4i_case_study/seminar_submission.csv


,part_id,machine_failure
0,EVAL-00001,0.005651
1,EVAL-00002,0.005814
2,EVAL-00003,0.317294
3,EVAL-00004,0.012170
4,EVAL-00005,0.034646


## Необязательный самостоятельный эксперимент

Для самопроверки подготовьте краткий разбор:

1. объект, target, момент решения и возможная утечка;
2. сравнение минимум трёх моделей на одинаковой CV;
3. объяснение, почему итоговая модель сильнее baseline;
4. два порога и то, как они меняют производственное решение;
5. валидный submission.

**Бонус:** найдите один уверенный ошибочный OOF-прогноз и предложите гипотезу, почему модель ошиблась.